# 02 模块验证 · 位流 / DMA / xRFDC / LMK

B227DR（xczu27dr-ffvf1517-2-i）· PYNQ 4.0 · 板卡 `192.168.1.5` · Jupyter 以 root 跑在 9090。

本册只做**模块级验证**，不写应用逻辑：每一步都输出 `PASS / FAIL / SKIP`，最后一张汇总表。
不发波形、不做环回、不做频谱 —— 那是 01 号册的事。

| 步 | 验证对象 | 判据（硬） |
|---|---|---|
| 1 | `soc.bit` 位流 | MD5 一致 + 4 个 IP 在册且基地址对得上 |
| 2 | AXI DMA ×2 | DMASR 可读、复位/RS 语义正确、MM2S 真搬完、S2MM canary 被覆盖 |
| 3 | xRFDC | `is_ready`、fs≈3932.16、6 tile 全 `0xF` + PLL 锁、FIFO_EN 使能、NCO=960 MHz |
| 4 | LMK04828 | SPI 读回可信、PLL1/PLL2 锁定、与 TICS 表整表一致、SYSREF 连续 |

## 执行顺序（冷启动必看）

1 → 2 → **4** → 3 → 4（复验）。

冷启动后 LMK 处于 POR 态，PLL2 不锁 ⇒ RFDC 的 tile 不可能 `0xF`。
所以第 3 步先跑一遍看基线、**允许它 FAIL**，把第 4 步做完（默认只读验证；确认要写表时把
`DO_PROGRAM` 改成 `True`），再回第 3 步复验，最后回第 4 步确认锁定与 tile 双绿。

In [1]:
import os
import re
import sys
import time
import platform

import numpy as np

CHECKS = []


def check(name, ok, detail="", skip=False, hint=""):
    """登记一项检查。skip=True 时不计入失败（依赖未满足，本轮无法判定）。"""
    tag = "SKIP" if skip else ("PASS" if ok else "FAIL")
    CHECKS.append({"name": name, "tag": tag, "detail": detail, "hint": hint})
    print("  [%s] %-46s %s" % (tag, name, detail))
    return ok


def summary(title="汇总"):
    rows = {}
    for c in CHECKS:                       # 同名项只留最后一次（重跑某一步不会堆积）
        rows[c["name"]] = c
    rows = list(rows.values())
    print("\n" + "=" * 80)
    print(" " + title)
    print("=" * 80)
    print("  %-5s %-48s %s" % ("结果", "检查项", "实测 / 说明"))
    print("  " + "-" * 78)
    for c in rows:
        print("  %-5s %-48s %s" % (c["tag"], c["name"], c["detail"]))
    n = {"PASS": 0, "FAIL": 0, "SKIP": 0}
    for c in rows:
        n[c["tag"]] += 1
    print("  " + "-" * 78)
    print("  PASS %d / FAIL %d / SKIP %d" % (n["PASS"], n["FAIL"], n["SKIP"]))
    bad = [c for c in rows if c["tag"] == "FAIL"]
    if bad:
        print("\n  未通过项与处置：")
        for c in bad:
            print("   - %-46s %s" % (c["name"], c["hint"] or c["detail"]))
    return bad


print("kernel :", platform.release())
print("python :", sys.version.split()[0])
print("cwd    :", os.getcwd())
root = (os.geteuid() == 0)
check("运行身份是 root（/dev/mem、sysfs、spidev 都要）", root, "uid=%d" % os.geteuid(),
      hint="Jupyter 必须以 root 起；否则 MMIO 与 SPI 全部打不开")

kernel : 6.12.40-xilinx-g0900176227cc
python : 3.12.3
cwd    : /home/xilinx/jupyter_notebooks
  [PASS] 运行身份是 root（/dev/mem、sysfs、spidev 都要）           uid=0


True

## 1. 装载 bit 流

`Overlay` 会按 `soc.bit` 找同名 `soc.hwh`；两者必须同目录同基名，否则 PYNQ 直接抛
`AttributeError: 'NoneType' object has no attribute 'items'`（它的报错与真实原因无关）。

MD5 取自本地原件（`soc.bit` 14 025 783 B / `soc.hwh` 960 270 B）。不一致说明传输被截断，
后面所有现象都没有意义 —— 这是最省时间的一道闸。

In [2]:
BIT = "/home/xilinx/soc_test/soc.bit"
HWH = "/home/xilinx/soc_test/soc.hwh"
WANT_MD5 = {
    "soc.bit": "95ce908385174d86676b84db77e172c1",
    "soc.hwh": "23eb351a4c663a469dc23354f504bcff",
}

import hashlib

for p in (BIT, HWH):
    name = os.path.basename(p)
    if not os.path.exists(p):
        check("文件存在 %s" % name, False, "缺失 " + p,
              hint="先上传 soc.bit / soc.hwh 到 /home/xilinx/soc_test/")
        continue
    h = hashlib.md5(open(p, "rb").read()).hexdigest()
    check("文件存在 %s" % name, True, "%d B" % os.path.getsize(p))
    check("MD5 一致 %s" % name, h == WANT_MD5[name], h[:16] + "...",
          hint="传输被截断或被替换过，不要往下跑")

from pynq import Overlay

t0 = time.time()
ol = Overlay(BIT)
check("Overlay 加载成功", True, "%.0f ms，%d 个 IP" % ((time.time() - t0) * 1e3, len(ol.ip_dict)))

EXPECT = [
    ("axi_dma_0",               0x80000000, "S2MM 收"),
    ("bkt_forwarder_0",         0x80010000, "慢控隧道"),
    ("axi_dma_1",               0x80020000, "MM2S 发"),
    ("usp_rf_data_converter_0", 0x80040000, "RFDC"),
]


def find_ip(suffix):
    """按 hwh 实例名匹配：key 可能是 "bb/axi_dma_0"，也可能是 "rf_adda_phy_<名>"。"""
    for k, v in ol.ip_dict.items():
        tail = k.split("/")[-1]
        if tail == suffix or tail.endswith("_" + suffix) or k == suffix:
            return k, v.get("phys_addr")
    return None, None


print()
for suffix, want, note in EXPECT:
    k, got = find_ip(suffix)
    check("IP 在册 %-26s %s" % (suffix, note), k is not None,
          ("0x%08X" % got) if got is not None else "MISSING",
          hint="位流与 hwh 不同源，或加载了别的位流")
    if got is not None:
        check("基地址 %-26s %s" % (suffix, note), got == want,
              "0x%08X（期望 0x%08X）" % (got, want),
              hint="与 README §2 的地址表不符：驱动会打到别的从机")

  [PASS] 文件存在 soc.bit                                   14025783 B
  [PASS] MD5 一致 soc.bit                                 95ce908385174d86...
  [PASS] 文件存在 soc.hwh                                   960270 B
  [PASS] MD5 一致 soc.hwh                                 23eb351a4c663a46...


  [PASS] Overlay 加载成功                                   2012 ms，5 个 IP

  [PASS] IP 在册 axi_dma_0                  S2MM 收        0x80000000
  [PASS] 基地址 axi_dma_0                  S2MM 收          0x80000000（期望 0x80000000）
  [PASS] IP 在册 bkt_forwarder_0            慢控隧道          0x80010000
  [PASS] 基地址 bkt_forwarder_0            慢控隧道            0x80010000（期望 0x80010000）
  [PASS] IP 在册 axi_dma_1                  MM2S 发        0x80020000
  [PASS] 基地址 axi_dma_1                  MM2S 发          0x80020000（期望 0x80020000）
  [PASS] IP 在册 usp_rf_data_converter_0    RFDC          0x80040000
  [PASS] 基地址 usp_rf_data_converter_0    RFDC            0x80040000（期望 0x80040000）


## 2. 验证 DMA

两颗 AXI DMA 都是**简单模式**（`C_INCLUDE_SG=0`），只有一组控制/状态寄存器：

| 检查 | 手段 | 期望 |
|---|---|---|
| AXI-Lite 通路 | 读 `DMASR` | 不等于 `0xFFFFFFFF`（读到全 1 = 打到空地址） |
| 软复位语义 | `DMACR.Reset(bit2)` 自清、`DMASR.Halted(bit0)` 置起 | Halted=1 |
| Run/Stop | `DMACR.RS(bit0)=1` | Halted 清零 |
| 硬件参数 | `hwh` 的 `C_SG_LENGTH_WIDTH` | 与驱动的 14 bit / 4095 字上限一致 |
| MM2S 真搬运 | 256 字 → 等 `DMASR.IOC(bit4)` | 完成且无 Err/IntErr/SlvErr/DecErr |
| S2MM 真落盘 | canary `0xDEADBEEF` 预填 → 收 256 字 | 至少一个字被改写 |

`S2MM` 依赖 ADC 有数据流，**冷启动时必然是 timeout** —— 那条记为 SKIP，等第 3 步 RFDC 起来后
重跑本格即可闭环。别把 SKIP 当成 DMA 坏。

`C_SG_LENGTH_WIDTH=14` 是硬约束：单次最多 16383 B = 4095 个 32 bit 字 = 16.66 µs @245.76 MSPS。

In [3]:
from pynq import MMIO, allocate

sys.path.insert(0, "/home/xilinx/soc_test")
from sdr1x1 import DmaChannel, SR_W1C, LEN_MAX, MAX_WORDS   # 复用已验证的寄存器封装

DMA = {
    "RX(S2MM)": DmaChannel(MMIO(0x80000000, 0x10000), "s2mm"),
    "TX(MM2S)": DmaChannel(MMIO(0x80020000, 0x10000), "mm2s"),
}

# ---- 2.1 寄存器可达 -------------------------------------------------------
print("=== 2.1 AXI-Lite 可达性 ===")
for tag, ch in DMA.items():
    raw = ch.mmio.read(ch.r_dmasr) & 0xFFFFFFFF
    check("%s DMASR 可读（≠0xFFFFFFFF）" % tag, raw != 0xFFFFFFFF, "0x%08X" % raw,
          hint="MMIO 打到空地址：位流没加载 / 基地址不对")

# ---- 2.2 软复位 -----------------------------------------------------------
print("\n=== 2.2 软复位与 Halted ===")
for tag, ch in DMA.items():
    try:
        st = ch.reset().status()
        check("%s 复位后 Halted=1" % tag, st["halted"], st["decoded"],
              hint="DMACR.Reset 未自清：AXI-Lite 时钟没到")
    except Exception as e:
        check("%s 复位后 Halted=1" % tag, False, "异常：%s" % e)

# ---- 2.3 Run/Stop ---------------------------------------------------------
print("\n=== 2.3 DMACR.RS ===")
for tag, ch in DMA.items():
    ch.mmio.write(ch.r_dmacr, 0x1)
    time.sleep(0.002)
    st = ch.status()
    ch.mmio.write(ch.r_dmacr, 0x0)
    check("%s RS=1 后脱离 Halted" % tag, not st["halted"], st["decoded"],
          hint="RS 写不进去，基本等于位流/地址不对")

# ---- 2.4 硬件参数与驱动假设一致 -------------------------------------------
print("\n=== 2.4 hwh 参数 vs 驱动常量 ===")
try:
    params = ol.ip_dict[find_ip("axi_dma_0")[0]].get("parameters", {})
    sg = params.get("C_SG_LENGTH_WIDTH")
except Exception as e:
    sg = "读取失败：%s" % e
check("hwh C_SG_LENGTH_WIDTH = 14", str(sg) == "14", "读到 %s" % sg,
      hint="改过 IP 就必须同步改驱动的 LEN_MAX")
check("驱动上限 %d B = %d 字" % (LEN_MAX, MAX_WORDS), LEN_MAX == (1 << 14) - 1,
      "%d B / %d 字" % (LEN_MAX, MAX_WORDS))

# ---- 2.5 MM2S 单次搬运 ----------------------------------------------------
print("\n=== 2.5 MM2S 单次搬运（TX 侧不依赖 ADC）===")
tx = DMA["TX(MM2S)"]
tbuf = allocate(shape=(256,), dtype=np.uint32)
tbuf[:] = (np.arange(256, dtype=np.uint32) * 0x00010001 + 0x12340000)
tbuf.flush()
try:
    tx.reset()
    tx.arm(int(tbuf.device_address), int(tbuf.nbytes))
    ok_tx, st = tx.wait(0.2)
    tx.stop()
    check("MM2S 搬运 256 字完成", ok_tx, "via=%s | %s" % (st["via"], st["decoded"]),
          hint="tready 悬空时靠 diver_demo 的组合逻辑也能吞，若整体 timeout 查第 1 步地址")
    check("MM2S 无错误位", not st["error"], st["decoded"],
          hint="DecErr/IntErr 多半是源地址越界（PYNQ 缓冲区没 flush）")
except Exception as e:
    check("MM2S 搬运 256 字完成", False, "异常：%s" % e)

# ---- 2.6 S2MM canary ------------------------------------------------------
print("\n=== 2.6 S2MM canary（依赖 ADC 出数）===")
rx = DMA["RX(S2MM)"]
rbuf = allocate(shape=(256,), dtype=np.uint32)
CANARY = 0xDEADBEEF
rbuf[:] = np.uint32(CANARY)
rbuf.flush()
try:
    rx.reset()
    rx.arm(int(rbuf.device_address), int(rbuf.nbytes))
    ok_rx, st = rx.wait(0.3)
    rx.stop()
    rbuf.invalidate()
    hit = int(np.sum(np.asarray(rbuf) != np.uint32(CANARY)))
    if ok_rx:
        check("S2MM canary 被覆盖", hit > 0, "%d/%d 字被改写" % (hit, rbuf.size),
              hint="DMA 完成但一个字节没动：FIFO 被禁用或 ADC 无时钟")
    else:
        check("S2MM canary 被覆盖", False, "timeout，canary 全在（%d 字）" % rbuf.size,
              skip=True, hint="冷启动正常：ADC 还没时钟。第 3 步 PASS 后重跑本格即闭环")
except Exception as e:
    check("S2MM canary 被覆盖", False, "异常：%s" % e, skip=True)

=== 2.1 AXI-Lite 可达性 ===
  [PASS] RX(S2MM) DMASR 可读（≠0xFFFFFFFF）                 0x00000001
  [PASS] TX(MM2S) DMASR 可读（≠0xFFFFFFFF）                 0x00000001

=== 2.2 软复位与 Halted ===
  [PASS] RX(S2MM) 复位后 Halted=1                          Halted
  [PASS] TX(MM2S) 复位后 Halted=1                          Halted

=== 2.3 DMACR.RS ===
  [PASS] RX(S2MM) RS=1 后脱离 Halted                       (全 0)
  [PASS] TX(MM2S) RS=1 后脱离 Halted                       Idle, IOC_IrqEn

=== 2.4 hwh 参数 vs 驱动常量 ===
  [PASS] hwh C_SG_LENGTH_WIDTH = 14                     读到 14
  [PASS] 驱动上限 16383 B = 4095 字                          16383 B / 4095 字

=== 2.5 MM2S 单次搬运（TX 侧不依赖 ADC）===
  [PASS] MM2S 搬运 256 字完成                                via=idle | Idle, IOC_IrqEn
  [PASS] MM2S 无错误位                                      Idle, IOC_IrqEn

=== 2.6 S2MM canary（依赖 ADC 出数）===
  [PASS] S2MM canary 被覆盖                                256/256 字被改写


## 3. 验证 xRFDC

`xrfdc` 是本板自研的**扁平 ctypes 封装**（`/usr/local/share/pynq-venv/lib/python3.12/site-packages/xrfdc.py`），
不是 PYNQ 插件 —— `ol.rfdc` 恒为 `None`，别去找 `RFdc()`。

**`xrfdc.init()` 内含 StartUp，重复调用会把 ADC tile 拖到 `0xC`**，所以只在 `is_ready()` 为假时初始化一次。

| 检查 | 期望 | 说明 |
|---|---|---|
| `is_ready()` | True | root + 位流已载 |
| `fs_mhz()` | ≈ 3932.16 MSPS | hwh 声明 4096，实际 refclk 只有 245.76 ⇒ 等比 ×0.96 |
| `fs/16` | ≈ 245.76 MSPS | fabric / DMA 的字流速率，波形换算一律用它 |
| 6 个 tile | `enabled=1` `state=0xF` `pll=1` | 非 FULL 也能出数，但 `0xF` 才是完全体 |
| `FIFO_EN(0x230)` | bit[1:0] = 0 | **低有效禁用位**；写 0x1/0x1F 反而是关 |
| NCO(ADC0/DAC0 blk0) | 960 MHz | C2R 上搬、R2C 下搬，同一中频才互相抵消 |

tile 基址：ADC0 `0x14000`、DAC0 `0x4000`，步进 `0x4000`，DRP 区在 `+0x2000`。

In [4]:
import xrfdc

print("xrfdc 模块:", xrfdc.__file__)
try:
    if not xrfdc.is_ready():
        xrfdc.init()
        print("  is_ready 为假 → 已 init()（含 StartUp）")
    else:
        print("  is_ready 已为真 → 跳过 init()，避免二次 StartUp")
    ready = xrfdc.is_ready()
except Exception as e:
    ready = False
    print("  xrfdc 异常:", e)
check("xrfdc.is_ready()", ready, "ready=%s" % ready,
      hint="非 root / 位流未加载；不要靠反复 init 硬试")

# ---- 3.1 采样率 -----------------------------------------------------------
print("\n=== 3.1 采样率 ===")
try:
    fs_rf = float(xrfdc.fs_mhz())
    fs_fab = fs_rf / 16.0
    check("ADC fs_mhz() ≈ 3932.16 MSPS", abs(fs_rf - 3932.16) < 8.0, "%.3f MSPS" % fs_rf,
          hint="LMK 频点不是 245.76 表，或 RFDC PLL 参数被改过")
    check("fabric/DMA 字率 fs/16 ≈ 245.76", abs(fs_fab - 245.76) < 0.5, "%.3f MSPS" % fs_fab)
    print("  pll_fs_mhz(ADC0) = %.4f   ← 寄存器按 256 参考配置的值，未修正"
          % float(xrfdc.pll_fs_mhz(1, 0)))
except Exception as e:
    check("ADC fs_mhz() ≈ 3932.16 MSPS", False, "异常：%s" % e)

# ---- 3.2 tile 状态 --------------------------------------------------------
print("\n=== 3.2 tile 状态 ===")
rows = []
try:
    for is_adc, kind, ntile in ((1, "ADC", 4), (0, "DAC", 2)):
        for t in range(ntile):
            st = xrfdc.tile_status(is_adc, t)
            rows.append(st)
            print("  %s%d  enabled=%d  state=0x%-3X  pll=%d  powerup=%d"
                  % (kind, t, st["enabled"], st["state"], st["pll"], st["powerup"]))
except Exception as e:
    print("  读取 tile 失败：", e)

if rows:
    n, nen = len(rows), sum(1 for s in rows if s["enabled"])
    nfull = sum(1 for s in rows if s["state"] == 0xF)
    npll = sum(1 for s in rows if s["pll"] == 1)
    check("%d 个 tile 全部 enabled" % n, nen == n, "%d/%d" % (nen, n))
    check("%d 个 tile 全部 state=0xF(FULL)" % n, nfull == n, "%d/%d" % (nfull, n),
          hint="非 FULL 仍可能出数；若同时 PLL 全灭，先做第 4 步")
    check("%d 个 tile 全部 PLL 锁定" % n, npll == n, "%d/%d" % (npll, n),
          hint="PLL 未锁 = 时钟上游问题（LMK/SYSREF），不是 RFDC 本身坏")

# ---- 3.3 FIFO 使能（低有效）----------------------------------------------
print("\n=== 3.3 FIFO_EN(0x230) bit[1:0] 必须为 0 ===")
RF = MMIO(0x80040000, 0x40000)
bad_fifo = []
for kind, base, ntile in (("ADC", 0x14000, 4), ("DAC", 0x4000, 2)):
    for t in range(ntile):
        c = base + t * 0x4000
        v = RF.read(c + 0x230) & 0xFFFF
        st = RF.read(c + 0x0C) & 0xF
        print("  %s%d  FIFO_EN=0x%04X  CUR_STATE=0x%X" % (kind, t, v, st))
        if (v & 0x3) != 0:
            bad_fifo.append("%s%d=0x%04X" % (kind, t, v))
check("FIFO 全使能（6 个 tile）", not bad_fifo,
      ", ".join(bad_fifo) if bad_fifo else "6/6 bit[1:0]=0",
      hint="写 0x230=0x0000 清禁用位，再走 0x08/0x04 的 STATE0→15 序列")

# ---- 3.4 NCO --------------------------------------------------------------
print("\n=== 3.4 NCO / Mixer（tile0 blk0）===")
for is_adc, kind in ((1, "ADC"), (0, "DAC")):
    try:
        f, mode, mtype, coarse, evt = xrfdc.get_nco_mhz(is_adc, 0, 0)
        print("  %s0  NCO=%.3f MHz  mode=%d  type=%d  coarse=%d  evt=%d"
              % (kind, f, mode, mtype, coarse, evt))
        check("%s0 NCO = 960.000 MHz" % kind, abs(f - 960.0) < 1.0, "%.3f MHz" % f,
              hint="C2R/R2C 必须同频抵消，否则基带信号被搬到别的频点")
    except Exception as e:
        check("%s0 NCO = 960.000 MHz" % kind, False, "异常：%s" % e)

xrfdc 模块: /usr/local/share/pynq-venv/lib/python3.12/site-packages/xrfdc.py
  is_ready 为假 → 已 init()（含 StartUp）
  [PASS] xrfdc.is_ready()                               ready=True

=== 3.1 采样率 ===
  [PASS] ADC fs_mhz() ≈ 3932.16 MSPS                    3932.160 MSPS
  [PASS] fabric/DMA 字率 fs/16 ≈ 245.76                   245.760 MSPS
  pll_fs_mhz(ADC0) = 4.0960   ← 寄存器按 256 参考配置的值，未修正

=== 3.2 tile 状态 ===
  ADC0  enabled=1  state=0xF    pll=1  powerup=1
  ADC1  enabled=1  state=0xF    pll=1  powerup=1
  ADC2  enabled=1  state=0xF    pll=1  powerup=1
  ADC3  enabled=1  state=0xF    pll=1  powerup=1
  DAC0  enabled=1  state=0xF    pll=1  powerup=1
  DAC1  enabled=1  state=0xF    pll=1  powerup=1
  [PASS] 6 个 tile 全部 enabled                            6/6
  [PASS] 6 个 tile 全部 state=0xF(FULL)                    6/6
  [PASS] 6 个 tile 全部 PLL 锁定                             6/6

=== 3.3 FIFO_EN(0x230) bit[1:0] 必须为 0 ===
  ADC0  FIFO_EN=0x0000  CUR_STATE=0xF
  ADC1  FIFO_EN=0x0000  CUR_STATE=0xF


## 4. 验证 LMK04828

LMK 只走 SPI（`/dev/spidev1.0`），与 PL 无关 —— 位流没加载也能验证。

读回要过三关，缺一不可：

1. **字节窗口自检**：借 `R0x142`（静态配置不用的动态延时寄存器）写互补值读回，确认 3 字节
   传输里 `rx[2]` 就是数据字节，不靠猜 POR 值。
2. **必须先 `configure()`**：无条件写 `R0x15F=0x13`（LD1 输出 PLL1 状态，**不要**改成 readback，
   本板两脚并联同一根 MISO，双推挽会互相打架）→ `R0x16E=0x3B`（LD2 读回，TYPE=3 推挽）。
3. **可信度闸门**：`R0x171` 必须是 `0xAA`、`R0x172` 必须是 `0x02`。这两个是固定寄存器，
   任何有效表下都不变；读不到就说明 MISO 上有第二个人在驱动，整表比对全是假差异。

过了闸门才读锁定：

- `R0x182` bit1 = PLL1 锁定（参考输入 / OSCin）
- `R0x183` bit1 = PLL2 锁定 ⇒ **这一位才是 RFDC 的时钟源**

最后与 TICS 表整表比对。板上有两张表（`245m76` 143 行 / `256m` 139 行），**两张都比**，
差异最少的那个就是芯片里当前的配置 —— 这比猜「到底该用哪张」可靠。

比对时另有 3 个地址**必然对不上，属预期**，已单独归类（`BENIGN`）：

| 地址 | 表内值 → 实读 | 原因 |
|---|---|---|
| `R0x000` | `0x90` → `0x10` | bit7 是 RESET，写进去后芯片自己清零 |
| `R0x143` | `0x11` → `0x53` | `sysref_start.py` 改 SYNC_MODE=3；表里的 0x11 在等 SYNC 引脚，本板没驱动它 |
| `R0x144` | `0x00` → `0x7F` | 同上，把其它分频器与 SYNC 隔离，免得参考时钟被一起复位 |

> **2026-11-10 实测结论**：芯片里是 **`256m` 表**（119 帧仅 3 条差异，全部是上表那三条）。
> `245m76` 表差 31 条（`DCLKout0_DIV` 是 `0x0C` 而表里 `0x0A`）—— 这张不是当前配置，别用。
> PLL2 已锁；PLL1 空载 `LD=0` 属正常，判据只看 PLL2 + tile FULL。

In [5]:
sys.path.insert(0, "/home/xilinx")
import lmk04828_readback as lrb

node = lrb.find_lmk_spi()
check("找到 ti,lmk04828 的 SPI 节点", node is not None, str(node),
      hint="设备树没使能 spi1 / compatible 不是 ti,lmk04828")

dev = None
try:
    dev = lrb.ensure_spidev(node)
    check("绑定 spidev（driver_override）", True, dev)
except Exception as e:
    check("绑定 spidev（driver_override）", False, "异常：%s" % e, hint="需要 root")

spi = dev and lrb.Spi(dev, speed=500000)
rbk = spi and lrb.Readback(spi, samples=5)

# ---- 4.1 读回通道 ---------------------------------------------------------
print("\n=== 4.1 读回通道自检 ===")
ok_win = False
ok_sane = False
if rbk is not None:
    try:
        ok_win = bool(rbk.probe_window())
    except Exception as e:
        print("  probe_window 异常:", e)
    check("写-读回环自检（窗口 = rx[2]）", ok_win, "0x142 写 0x05/0x0A 均读回一致",
          hint="SPI 速率过高或 MISO 未上拉；先降到 100k 试")

    print("\n=== 4.2 配置 readback 模式 ===")
    try:
        rbk.configure()
        ok_sane = bool(rbk.sanity())
    except Exception as e:
        print("  configure/sanity 异常:", e)
    check("读回可信（R0x171=0xAA / R0x172=0x02）", ok_sane,
          "通过" if ok_sane else "不可信",
          hint="R0x15F 不在 0x13 → Status_LD1 与 Status_LD2 抢 MISO，比对结果整片是假的")

# ---- 4.3 锁定 -------------------------------------------------------------
print("\n=== 4.3 PLL 锁定 ===")
ld1 = ld2 = 0
if rbk is not None:
    v1, _ = rbk.read(0x182)
    v2, _ = rbk.read(0x183)
    ld1 = ((v1 or 0) >> 1) & 1
    ld2 = ((v2 or 0) >> 1) & 1
    print("  PLL1: R0x182=0x%02X  LD=%d  LOST=%d" % (v1 or 0, ld1, ((v1 or 0) >> 2) & 1))
    print("  PLL2: R0x183=0x%02X  LD=%d  LOST=%d" % (v2 or 0, ld2, ((v2 or 0) >> 2) & 1))
    check("PLL1 锁定（参考输入 / OSCin）", ld1 == 1, "R0x182=0x%02X  LD=%d" % (v1 or 0, ld1),
          skip=(ld1 != 1),
          hint="本板参考是 19.2 MHz OSCin 直接进 PLL2，PLL1 空载时 LD1=0 属正常："
               "判据是 PLL2 锁定 + tile FULL，不是 LD1")
    check("PLL2 锁定（RFDC 的时钟源）", ld2 == 1, "R0x183=0x%02X  LD=%d" % (v2 or 0, ld2),
          skip=(not ok_sane), hint="PLL2 不锁 ⇒ 第 3 步 tile 永远上不了 0xF。写表见第 4.7 格")

# ---- 4.4 整表比对 ---------------------------------------------------------
print("\n=== 4.4 与 TICS 表整表比对 ===")

# 与表必然对不上的地址：写进去之后芯片自己会改，或本流程故意改过
BENIGN = {
    0x000: "bit7=RESET 写后自清：表写 0x90，实读 0x10 属正常",
    0x143: "sysref_start.py 写 0x53 启 SYNC_MODE=3（表里 0x11 在等 SYNC 引脚，永远等不到）",
    0x144: "sysref_start.py 写 0x7F 隔离其它分频器，避免 SYNC 把参考时钟也复位",
}


def cmp_tics(path):
    exp = lrb.parse_tics(path)
    n = 0
    mism, benign = [], []
    for addr, want in exp:
        if addr in lrb.LIVE_REGS:          # 只读/实时状态寄存器不参与
            continue
        n += 1
        got, _ = rbk.read(addr)
        if got == want:
            continue
        if addr in BENIGN:
            benign.append((addr, want, got))
        else:
            mism.append((addr, want, got))
    return n, mism, benign


res = []
if ok_sane:
    for path in ("/home/xilinx/LMK04828_245m76.tics.txt",
                 "/home/xilinx/LMK04828_256m.tics.txt"):
        if not os.path.exists(path):
            print("  跳过（文件不存在）：", path)
            continue
        n, mism, benign = cmp_tics(path)
        name = os.path.basename(path)
        res.append((name, n, mism, benign))
        print("  --- %s：%d 帧参与，%d 条不一致（另有 %d 条可解释）---"
              % (name, n, len(mism), len(benign)))
        for addr, want, got in mism[:10]:
            print("      R0x%03X  表=0x%02X  实读=%s"
                  % (addr, want, ("0x%02X" % got) if got is not None else "--"))
        if len(mism) > 10:
            print("      ... 另有 %d 条" % (len(mism) - 10))
        for addr, want, got in benign:
            print("      R0x%03X  表=0x%02X  实读=0x%02X   ← %s"
                  % (addr, want, got, BENIGN[addr]))

if res:
    # 只对「最接近」的那张表下判决；另一张本来就不是当前配置，不算失败
    best = min(res, key=lambda r: len(r[2]))
    check("LMK 配置与 %s 一致" % best[0], not best[2],
          "%d/%d 条差异（%d 条已解释）" % (len(best[2]), best[1], len(best[3])),
          hint=("零星差异：看是不是 0x143/0x144（SYSREF 启动时改的）或 0x000（RESET 自清）"
                if len(best[2]) <= 5 else
                "差异成片（>10 条）：先怀疑读回不可信，再怀疑表没写进去"))
    for name, n, mism, benign in res:
        if name != best[0]:
            print("  对照：%s 有 %d 条差异 —— 该表不是芯片当前配置，属正常" % (name, len(mism)))
    print("\n  → 芯片当前配置 = %s（其余表差异 %s）"
          % (best[0], ", ".join("%s:%d" % (r[0].split('_')[1][:6], len(r[2]))
                                for r in res if r[0] != best[0]) or "无"))
else:
    print("  读回不可信，整表比对无意义，已跳过")

# ---- 4.5 SYSREF -----------------------------------------------------------
print("\n=== 4.5 SYSREF 现状 ===")
if rbk is not None:
    mux = (rbk.read(0x139)[0] or 0)
    divh = (rbk.read(0x13A)[0] or 0)
    divl = (rbk.read(0x13B)[0] or 0)
    sync = (rbk.read(0x143)[0] or 0)
    pulse = (rbk.read(0x13E)[0] or 0)
    div = ((divh & 0x1F) << 8) | divl
    modes = {0: "Normal SYNC", 1: "Re-clocked", 2: "SYSREF pulser", 3: "SYSREF continuous"}
    print("  R0x139=0x%02X  SYSREF_MUX=%d (%s)"
          % (mux, mux & 3, modes.get(mux & 3, "?")))
    print("  R0x13A/B → SYSREF_DIV=%d" % div)
    print("  R0x143=0x%02X  SYNC_MODE=%d  EN=%d  1SHOT=%d"
          % (sync, sync & 3, (sync >> 4) & 1, (sync >> 6) & 1))
    print("  R0x13E=0x%02X  SYSREF_PULSE_CNT=%d" % (pulse, pulse))
    check("SYSREF_MUX = 3（continuous）", (mux & 3) == 3, "0x%02X" % mux,
          hint="跑 sudo python3 /home/xilinx/sysref_start.py --sysref-div 384")

# ---- 4.6 与 RFDC 交叉验证 -------------------------------------------------
print("\n=== 4.6 交叉验证：SYSREF 真的到 RFDC 了吗 ===")
try:
    st = xrfdc.tile_status(1, 0)
    check("ADC0 tile 已 FULL（SYSREF + 时钟闭环）", st["state"] == 0xF,
          "state=0x%X pll=%d" % (st["state"], st["pll"]),
          hint="LMK 全绿但 tile 不 FULL：SYSREF 分频器没起振（R0x143 SYNC_MODE 不是 3）")
except Exception as e:
    check("ADC0 tile 已 FULL（SYSREF + 时钟闭环）", False, "xrfdc 不可用：%s" % e, skip=True)

if spi is not None:
    spi.close()

  [PASS] 找到 ti,lmk04828 的 SPI 节点                        /sys/bus/spi/devices/spi1.0
  [PASS] 绑定 spidev（driver_override）                     /dev/spidev1.0

=== 4.1 读回通道自检 ===
  读回窗口自检通过（3 字节传输，rx[2] = 数据字节）
  [PASS] 写-读回环自检（窗口 = rx[2]）                            0x142 写 0x05/0x0A 均读回一致

=== 4.2 配置 readback 模式 ===

=== 配置四线 SPI readback ===
  SPI_LOCK 解锁 (R0x1FFD/R0x1FFE=0x00, R0x1FFF=0x53)
  R0x000 = 0x10 (SPI_3WIRE_DIS=1)
  R0x15F 无条件写回 0x13（写前读到 0x13，脏状态下该读数不可信）
  R0x16E = 0x3B (PLL2_LD_MUX=7 readback, TYPE=3 推挽)
  读回可信度检查通过（R0x171=0xAA / R0x172=0x02）
  [PASS] 读回可信（R0x171=0xAA / R0x172=0x02）                通过

=== 4.3 PLL 锁定 ===
  PLL1: R0x182=0x00  LD=0  LOST=0
  PLL2: R0x183=0x02  LD=1  LOST=0
  [SKIP] PLL1 锁定（参考输入 / OSCin）                          R0x182=0x00  LD=0
  [PASS] PLL2 锁定（RFDC 的时钟源）                             R0x183=0x02  LD=1

=== 4.4 与 TICS 表整表比对 ===
  --- LMK04828_245m76.tics.txt：119 帧参与，28 条不一致（另有 3 条可解释）---
      R0x100  表=0x0A  实读=0x0C
      R0x101  表=0x55  实读=0x6

In [ ]:
# 4.7 可选：写入 TICS 表（默认关闭，模块验证本身只读）
DO_PROGRAM = False
TICS_TO_PROGRAM = "/home/xilinx/LMK04828_256m.tics.txt"   # 实测本板用 256m（见 4.4 比对结果）
SYSREF_DIV = 384

if not DO_PROGRAM:
    print("DO_PROGRAM = False —— 本格不写任何寄存器。")
    print("第 4.3 步 PLL2 未锁、或 4.4 差异成片时，才把 DO_PROGRAM 改成 True 重跑本格。")
    print("写完后：回第 4 步第 1 格复读锁定 → 回第 3 步复验 tile → 回第 2 步复跑 S2MM canary。")
else:
    print(os.popen("python3 /home/xilinx/lmk04828_set.py --regs %s" % TICS_TO_PROGRAM).read())
    print(os.popen("python3 /home/xilinx/sysref_start.py --sysref-div %d" % SYSREF_DIV).read())
    print("寄存器表已写入。请按上面的顺序复验。")

## 5. 汇总

一张表回答「四块模块各自到底通没通」。`SKIP` 不等于失败 —— 它表示**依赖未满足、本轮无法判定**
（最典型：RFDC 没起来时 S2MM 收不到数）。

In [ ]:
bad = summary("02 模块验证汇总")

print()
if not bad:
    print("  四步全绿：位流装载、两颗 DMA、xRFDC、LMK04828 的模块级通路都通。")
else:
    print("  按上面的『未通过项与处置』逐条处理。上游未配置导致的失败，先修上游再回跑对应格。")

for b in (tbuf, rbuf):
    try:
        b.freebuffer()
    except Exception:
        pass
print("\n  临时缓冲区已释放；DMA 已 stop，DAC 停在最后一个字上（直流）。")